In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import seaborn as sns
import os, sys

sys.path.insert(0, os.path.abspath(".."))

from src.constants import COLOR_EXP as COLOR_0, COLOR_REC as COLOR_1, SHADE_COL, SHADE_A, LW
from src.data_loader import load_macro_data, load_asset_data
from src.preprocess import prepare_macro_data, train_test_split
from src.hmm_model import HMMRegimeModel
from src.plots import (plot_regime_states, plot_regime_probs,
                        plot_series_colored, plot_series_shaded,
                        plot_frontiers, plot_backtest, plot_regime_nber, _fmt_matrix)
from src.portfolio import (estimate_regime_moments, optimize_regimes,
                            efficient_frontier, shrinkage_diagnostics)
from src.metrics import compute_metrics
from src.backtest import run_backtest, print_metrics_table

# Backward-compat wrappers
def fit_hmm(X, K=2, n_iter=200, n_restarts=10, tol=1e-6, random_state=42):
    return HMMRegimeModel(n_states=K, n_iter=n_iter, n_restarts=n_restarts,
                           tol=tol, random_state=random_state).fit(X)

def get_smoothed_probs(model, X): return model.smoothed_probs(X)
def get_filtered_probs(model, X): return model.filtered_probs(X)

plot_hmm_states      = lambda d, s, title="HMM Regime States": plot_regime_states(d, s, title)
plot_hmm_state_probs = lambda d, p, title="HMM State Probabilities": plot_regime_probs(d, p, title)

## Description

- Goal: implement MVO to determine optimal portfolio weights with state-conditional asset returns and covariance matrix
- Methodology:
  - Step 1: Use 13 macroeconomic indicators to classify states (K=2)
  - Step 2: Estimate state-conditional mean and covariance (Ledoit-Wolf shrinkage)
  - Step 3: Implement MVO for each state to determine optimal portfolio weights
  - Step 4: Compare with static portfolios and backtest on historical test data

This notebook implements Steps 1-4 for the **Gaussian HMM** using `hmmlearn`.

## Data preparation

Standardize macroeconomic indicators and prepare train/test arrays.

In [ ]:
macro_df = load_macro_data("../data/macro_clean.csv")
macro_df.head()

In [ ]:
macro_df.info()

In [ ]:
X_train, X_test, scaler, dates_train, dates_test = prepare_macro_data(macro_df)

In [ ]:
X_train.shape, X_test.shape if X_test is not None else None

## Model fitting

In [ ]:
model = fit_hmm(X_train, K=2, n_iter=200, n_restarts=10, tol=1e-6, random_state=42)

## Extracting results

In [ ]:
smoothed_probs_train = get_smoothed_probs(model, X_train)
print("Smoothed probs shape:", smoothed_probs_train.shape)
print("Sum to 1:", np.allclose(smoothed_probs_train.sum(axis=0), 1.0))

In [ ]:
filtered_probs_train = get_filtered_probs(model, X_train)
print("Filtered probs shape:", filtered_probs_train.shape)
print("Sum to 1:", np.allclose(filtered_probs_train.sum(axis=0), 1.0))

In [ ]:
states_sequence = model.predict(X_train)
print("State counts:", np.unique(states_sequence, return_counts=True))

In [ ]:
states_sequence_series = pd.Series(states_sequence, index=dates_train, name="HMM State")
states_sequence_series.value_counts()

### Transition matrix

In [ ]:
pd.DataFrame(model.transmat_,
             columns=[f"State {i}" for i in range(model.n_components)],
             index=[f"State {i}" for i in range(model.n_components)])

In [ ]:
print(f"Average duration in each state (expected number of consecutive months):")
for i, persistent_prob in enumerate(model.transmat_.diagonal()):
    print(f"State {i}: {1 / (1 - persistent_prob):.2f} months")

### Conditional means and covariance matrices

In [ ]:
pd.DataFrame(model.means_,
             columns=macro_df.columns,
             index=["State 0 (Expansion)", "State 1 (Recession)"]).T

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.heatmap(model.covars_[0], annot=True, fmt=".1f", cmap="coolwarm",
            xticklabels=macro_df.columns, yticklabels=macro_df.columns)
plt.title("Covariance Matrix During Expansion (State 0)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(model.covars_[1], annot=True, fmt=".1f", cmap="coolwarm",
            xticklabels=macro_df.columns, yticklabels=macro_df.columns)
plt.title("Covariance Matrix in Recession (State 1)")
plt.tight_layout()
plt.show()

## Plotting results

In [ ]:
plot_hmm_states(dates_train, states_sequence, title="HMM Predicted States Over Time")
plt.show()

In [ ]:
plot_hmm_state_probs(dates_train, smoothed_probs_train,
                     title="HMM Smoothed State Probabilities Over Time")
plt.show()

In [ ]:
macro_df_expanded = macro_df.copy()
macro_df_expanded["HMM State"] = states_sequence

In [ ]:
asset_df = load_asset_data("../data/market_clean.csv")
asset_df.columns = ['Equity', 'Bonds', 'Gold']
asset_df_expanded = asset_df.copy()
asset_df_expanded["HMM State"] = macro_df_expanded["HMM State"]

In [ ]:
plot_series_colored(macro_df_expanded, series_name='vix', state_col='HMM State')
plt.show()

In [ ]:
for col in asset_df.columns:
    plot_series_shaded(asset_df_expanded, series_name=col, state_col="HMM State", ax=None)
    plt.show()

## Portfolio optimization

Regime-conditional MVO using Ledoit-Wolf shrinkage estimated on the training set.

In [ ]:
market_train, market_test = train_test_split(asset_df_expanded)
print("Last train date:", market_train.index[-1])
print("First test date:", market_test.index[0])

In [ ]:
moments = estimate_regime_moments(market_train,
                                  asset_cols=market_train.columns[:-1],
                                  state_col="HMM State")

In [ ]:
shrinkage_diagnostics(moments, asset_cols=market_train.columns[:-1])

In [ ]:
asset_names = market_train.columns[:-1]

print("=" * 55)
print("Regime-Conditional MVO")
print("=" * 55)
results = optimize_regimes(moments, asset_names, rf=0.0, long_only=False)

print("\n" + "=" * 55)
print("Efficient Frontiers")
print("=" * 55)
plot_frontiers(moments, asset_names, results)
plt.show()

### Target weights export

In [ ]:
output_path = os.path.join("..", "outputs/tables")
weights_by_regime = {
    k: results[k]['min_variance'].weights
    for k in results
}

rows = []
for date, regime in states_sequence_series.items():
    w = weights_by_regime[regime]
    rows.append({
        'Date':   date,
        'Regime': regime,
        'Equity': w[0],
        'Bonds':  w[1],
        'Gold':   w[2],
    })

df_target_weights = pd.DataFrame(rows).set_index('Date')
df_target_weights.to_csv(os.path.join(output_path, "hmm_target_weights.csv"))
df_target_weights.head()

### Backtesting

In [ ]:
regime_weights = {
    0: results[0]['min_variance'].weights,
    1: results[1]['min_variance'].weights,
}

backtest = run_backtest(
    df_test        = market_test,
    asset_cols     = market_test.columns[:-1].tolist(),
    state_col      = 'HMM State',
    regime_weights = regime_weights,
    equity_col     = 'Equity',
    bond_col       = 'Bonds',
    rf             = 0.0,
    freq           = 12,
    confidence     = 0.95,
    tc_bps         = 0.0,
)

print_metrics_table(backtest['metrics'])
plot_backtest(backtest, state_series=market_test['HMM State'])
plt.show()

### MVO (Max-Sharpe / Tangency) Portfolio

Compute the maximum Sharpe ratio (tangency) portfolio for each regime.

In [ ]:
from src.portfolio import MeanVariancePortfolio

mvo_weights_by_regime = {}
print('MVO (Tangency) Weights:')
print('=' * 55)
for k, m in moments.items():
    mvp = MeanVariancePortfolio(
        mu=m['mu'], Sigma=m['sigma'],
        asset_names=list(asset_names),
        rf=0.0, long_only=True, regime=k,
    )
    res = mvp.tangency()
    mvo_weights_by_regime[k] = res.weights
    print(f'  Regime {k}: Equity={res.weights[0]:.3f}  '
          f'Bonds={res.weights[1]:.3f}  Gold={res.weights[2]:.3f}  '
          f'Sharpe(ann)={res.sharpe * (12**0.5):.3f}')

rows_mvo = []
for date, regime in states_sequence_series.items():
    w = mvo_weights_by_regime[regime]
    rows_mvo.append({'Date': date, 'Regime': regime,
                     'Equity': w[0], 'Bonds': w[1], 'Gold': w[2]})

df_mvo_weights = pd.DataFrame(rows_mvo).set_index('Date')
df_mvo_weights.to_csv(os.path.join(output_path, 'hmm_mvo_target_weights.csv'))
print('\nSaved: hmm_mvo_target_weights.csv')
df_mvo_weights.head()

In [ ]:
from src.backtest import backtest_with_transaction_costs
from src.data_loader import load_asset_data
from src.constants import ASSET_COLS

asset_returns_full = load_asset_data('../data/market_clean.csv')[ASSET_COLS].sort_index()

mvo_test_weights = df_mvo_weights.rename(columns={
    'Equity': 'index_fund', 'Bonds': 'treasury_fund', 'Gold': 'gold_fund'
})

SIGNAL_START_HMM = pd.Timestamp('2018-10-31')
mvo_test_weights = mvo_test_weights.loc[mvo_test_weights.index >= SIGNAL_START_HMM].copy()

result_mvo = backtest_with_transaction_costs(
    target_weights=mvo_test_weights,
    asset_returns=asset_returns_full,
    transaction_cost_bps=10,
    apply_signal_lag=True,
    include_initial_trade_cost=False,
)

first_date = result_mvo.index[0]
result_mvo.loc[first_date, 'trade_notional']         = 0.0
result_mvo.loc[first_date, 'one_way_turnover']       = 0.0
result_mvo.loc[first_date, 'transaction_cost']       = 0.0
result_mvo.loc[first_date, 'net_return_after_costs'] = result_mvo.loc[first_date, 'gross_return']

result_mvo.to_csv(os.path.join(output_path, 'hmm_mvo_monthly_returns_with_costs.csv'))
print(f'Test period: {result_mvo.index.min().date()} to {result_mvo.index.max().date()}')

net_ret = result_mvo['net_return_after_costs']
geo_mean = (1 + net_ret).prod() ** (12 / len(net_ret)) - 1
vol = net_ret.std() * np.sqrt(12)
sharpe = (geo_mean - 0.02664) / vol
print(f'\nHMM-MVO (net of costs):')
print(f'  Ann. return (geo): {geo_mean*100:.2f}%')
print(f'  Ann. volatility  : {vol*100:.2f}%')
print(f'  Sharpe ratio     : {sharpe:.3f}')

## Regime-Shaded Indicator Plots: VIX, FEDFUNDS, and UNRATE

Line color reflects model-classified state; NBER recession bands provide an independent reference.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='vix', state_col='HMM State',
    recession_state=1,
    title='CBOE VIX: Model Regime vs NBER Recessions',
    ylabel='VIX (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='fed_funds_rate', state_col='HMM State',
    recession_state=1,
    title='Effective Federal Funds Rate: Model Regime vs NBER Recessions',
    ylabel='FEDFUNDS (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='unemployment_rate', state_col='HMM State',
    recession_state=1,
    title='Civilian Unemployment Rate: Model Regime vs NBER Recessions',
    ylabel='UNRATE (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

## Robustness Check: K = 3 Regimes

AIC and BIC are used for model comparison (LR test is non-standard under H0: K=2 per Garcia 1998).

In [ ]:
model_k3 = fit_hmm(X_train, K=3, n_iter=200, n_restarts=5, tol=1e-6, random_state=42)

In [ ]:
T, N = X_train.shape

def hmm_n_params(K, N):
    return (K - 1) + K * (K - 1) + K * N + K * N * (N + 1) // 2

ll_k2 = model.score(X_train)
ll_k3 = model_k3.score(X_train)

np_k2 = hmm_n_params(2, N)
np_k3 = hmm_n_params(3, N)

lr_stat = 2 * (ll_k3 - ll_k2)
delta_df = np_k3 - np_k2

summary_k3 = pd.DataFrame({
    'K': [2, 3],
    'Log-Likelihood': [ll_k2, ll_k3],
    'N Parameters':   [np_k2, np_k3],
    'AIC': [-2*ll_k2 + 2*np_k2, -2*ll_k3 + 2*np_k3],
    'BIC': [-2*ll_k2 + np_k2*np.log(T), -2*ll_k3 + np_k3*np.log(T)],
})
print(f'LR statistic (K=2 vs K=3): {lr_stat:.2f}  (delta_df = {delta_df})')
print('Note: LR p-value non-standard under H0 — use AIC/BIC.')
summary_k3

In [ ]:
states_k3 = model_k3.predict(X_train)
print('K=3 state counts:', dict(zip(*np.unique(states_k3, return_counts=True))))

means_k3 = pd.DataFrame(
    model_k3.means_,
    columns=macro_df.columns,
    index=[f'State {k}' for k in range(3)]
).T

key_vars = ['vix', 'credit_spread', 'unemployment_rate',
            'industrial_production', 'fed_funds_rate']
means_k3.loc[key_vars]

In [ ]:
summary_k3.to_csv(os.path.join(output_path, 'hmm_k3_model_selection.csv'), index=False)
means_k3.loc[key_vars].to_csv(os.path.join(output_path, 'hmm_k3_regime_means.csv'))
print('Saved K=3 HMM results.')